# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dushyant1607/ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: Random Forest Classifier**

I will use a Random Forest because it can capture nonlinear relationships between search visibility, CTR, page freshness, and engagement without assuming that their effects are linear. I will compare its ranking against the Week 4 CTR-opportunity rule using Precision@50 on the same held-out pages.

The target will be the dataset's declining-trend proxy, if the documented dataset field supports that definition. The target field and any label-derived or future-window fields will be excluded from model features. Results will be treated as directional decision-support, not proof that a page change will improve performance.

In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/ml-internship")
DATA = ROOT / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA)

# Binary target: declining vs not declining
df["target_down"] = (df["trend_direction"] == "down").astype(int)

# Exclude identifiers, target-derived fields, and recent-period
# measurements that directly contribute to the trend label.
excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target_down",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
}

features = [c for c in df.columns if c not in excluded]

# Use numeric and categorical predictors; keep missingness explicit.
X = df[features].copy()

for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].fillna("Missing").astype(str)

for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].replace([np.inf, -np.inf], np.nan)
    X[f"{col}_missing"] = X[col].isna().astype(int)
    X[col] = X[col].fillna(X[col].median())

y = df["target_down"]

print("Rows:", len(df))
print("Features:", len(X.columns))
print("\nTarget counts:")
print(y.value_counts().rename(index={0: "not_down", 1: "down"}))
print("\nTarget rate:", round(y.mean(), 4))
print("\nExcluded columns:", sorted(excluded))
print("\nFeature columns:")
print(X.columns.tolist())

assert "client_id" not in X.columns
assert "content_id" not in X.columns
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert not any(
    c in X.columns for c in [
        "impressions_last_30d", "impressions_prev_30d",
        "clicks_last_30d", "clicks_prev_30d",
        "sessions_last_30d", "sessions_prev_30d",
    ]
)

print("\nLeakage guard checks passed.")

Rows: 30000
Features: 57

Target counts:
target_down
down        16262
not_down    13738
Name: count, dtype: int64

Target rate: 0.5421

Excluded columns: ['clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'target_down', 'trend_direction', 'trend_pct']

Feature columns:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'search_volume_missing', 'competition_mi

## 2. Split design

**Split design: client-grouped holdout**

I will hold out 20% of clients for testing, using a fixed random seed. All pages belonging to a client will remain in the same partition, reducing leakage from client-specific patterns. The model and the Week 4 baseline will be evaluated on exactly the same held-out pages.

Precision@50 will be the primary ranking metric: the proportion of the 50 highest-ranked pages that belong to the declining class. The test set will not be used to fit the model or choose its parameters.

In [14]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"].fillna("missing_client").astype(str)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

assert train_clients.isdisjoint(test_clients), "Client leakage detected."
print("PASS: no client appears in both partitions.")

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Train positive rate: 0.5501
Test positive rate: 0.511
PASS: no client appears in both partitions.


## 3. Train + compare vs my baseline
**Model: Logistic Regression**

I will train Logistic Regression with preprocessing for numeric and categorical features. The model will predict the probability that a page is declining. I will compare its top-50 ranking with the Week 4 CTR-opportunity baseline on the same client-held-out test pages.

The primary metric is Precision@50. The comparison is directional because the target is a proxy derived from observed trend data, not a verified outcome of a content refresh.

In [15]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)

# Split using the indices already created in Section 2.
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [
    c for c in X_train.columns if c not in numeric_features
]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )),
])

model.fit(X_train, y_train)

# Model ranking scores
model_scores = model.predict_proba(X_test)[:, 1]

# Week 4 baseline: use the same CTR-gap and visibility logic.
test_rows = df.iloc[test_idx].copy()

reference_ctr = df.loc[
    (df["avg_position"] > 0)
    & (df["avg_position"] <= 10)
    & df["ctr"].notna(),
    "ctr",
].median()

test_ctr = pd.to_numeric(test_rows["ctr"], errors="coerce")
test_impressions = pd.to_numeric(
    test_rows["impressions_90d"], errors="coerce"
).fillna(0)

ctr_gap = (
    (reference_ctr - test_ctr).clip(lower=0) / reference_ctr
).fillna(0).clip(0, 1)

visibility = np.log1p(test_impressions)
visibility_max = visibility.max()

if visibility_max > 0:
    visibility = visibility / visibility_max

baseline_scores = (
    100 * ctr_gap * visibility
).fillna(0).to_numpy()

def precision_at_k(y_true, scores, k=50):
    k = min(k, len(y_true))
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

# Use exactly the same test rows and Precision@50 metric.
results = pd.DataFrame([
    {
        "method": "Week 4 CTR baseline",
        "Precision@50": precision_at_k(y_test, baseline_scores, 50),
    },
    {
        "method": "Logistic Regression",
        "Precision@50": precision_at_k(y_test, model_scores, 50),
    },
])

print("Test rows:", len(y_test))
print("Test positive rate:", round(float(y_test.mean()), 4))
print("\nModel vs baseline:")
print(results.to_string(index=False))

print("\nAdditional model metrics at threshold 0.5:")
predictions = (model_scores >= 0.5).astype(int)
print("Precision:", round(precision_score(y_test, predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, predictions, zero_division=0), 4))
print("F1:", round(f1_score(y_test, predictions, zero_division=0), 4))
print("Average precision:", round(average_precision_score(y_test, model_scores), 4))

# Save results for reproducibility.
output_path = ROOT / "work/outputs/w05_model_comparison.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(output_path, index=False)

print("\nSaved comparison:", output_path)

Test rows: 6163
Test positive rate: 0.511

Model vs baseline:
             method  Precision@50
Week 4 CTR baseline          0.68
Logistic Regression          0.60

Additional model metrics at threshold 0.5:
Precision: 0.558
Recall: 0.6446
F1: 0.5982
Average precision: 0.5639

Saved comparison: /content/ml-internship/work/outputs/w05_model_comparison.csv


## 4. Errors and interpretation

**Error analysis and interpretation**

I will inspect the highest-ranked predictions and the model's incorrect classifications. I will use permutation importance to identify which features contribute most to predictive performance. Feature importance indicates association, not causation.

False positives are pages predicted to be declining that are labelled otherwise. False negatives are declining pages the model misses. The label is a trend proxy, so disagreements may reflect noisy measurements or differences between historical performance signals and the derived label.

In [12]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix

# Predictions from the trained Section 3 model
predictions = (model_scores >= 0.5).astype(int)

errors = pd.DataFrame(
    {
        "actual": y_test.to_numpy(),
        "predicted": predictions,
        "predicted_probability": model_scores,
    },
    index=X_test.index,
)

errors["error_type"] = np.select(
    [
        (errors["actual"] == 0) & (errors["predicted"] == 1),
        (errors["actual"] == 1) & (errors["predicted"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, predictions))

print("\nError counts:")
print(errors["error_type"].value_counts())

print("\nThree examples of incorrect predictions:")
wrong = errors.loc[errors["error_type"] != "correct"].head(3).copy()

# Add anonymized measurements to help interpret each error.
review_cols = [
    c for c in [
        "content_type",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "content_age_days",
    ]
    if c in df.columns
]

wrong = wrong.join(df[review_cols], how="left")
print(wrong.to_string(index=True))

# Permutation importance on held-out data
importance = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=3,
    random_state=42,
    n_jobs=-1,
)

importance_table = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)

print("\nTop 10 permutation importances:")
print(importance_table.head(10).to_string(index=False))

print("\nInterpretation:")
print(
    "Permutation importance measures how predictive performance changes "
    "when a feature is shuffled. It indicates predictive contribution, "
    "not causation."
)


Confusion matrix [[TN, FP], [FN, TP]]:
[[1406 1608]
 [1119 2030]]

Error counts:
error_type
correct           3436
false_positive    1608
false_negative    1119
Name: count, dtype: int64

Three examples of incorrect predictions:
    actual  predicted  predicted_probability      error_type     content_type  impressions_90d   ctr  avg_position  days_since_last_update  content_age_days
13       0          1               0.751086  false_positive  keyword article              307  0.00          39.8                     103               238
23       1          0               0.196508  false_negative  keyword article              297  0.34          13.9                      20               502
26       0          1               0.580167  false_positive  keyword article             2426  0.12          30.0                      13               300

Top 10 permutation importances:
              feature  importance_mean  importance_std
days_with_impressions         0.037022        0.005616


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.